# Predicting tomorrow's air quality in Ohio: the simplest version, with feature scaling

**The idea:** look at **today** (air quality, weather, time of year) and guess **tomorrow's** air quality. Then check how good the guesses are on days the computer has never seen. This time we also **scale** the clues first.

**How to run:** choose **Runtime > Run all**. When the first cell asks, upload the Excel file `AQI_Ohio_2022_2026_with_weather.xlsx`. Everything runs in under a minute. What each column of the file means is explained in the Data Definition PDF (`AQI_Data_Definition.pdf`).

**Words you will see**
- **AQI** (Air Quality Index): higher means worse air. 0-50 is good, and above 100 is unhealthy for sensitive people.
- **Clue**: a fact we give the computer to help it guess, for example today's AQI.
- **Baseline**: the lazy guess "tomorrow will be the same as today". A model is only useful if it beats this.
- **MAE**: how many AQI points the guess is wrong by, on average. Smaller is better.
- **RMSE**: like MAE, but big mistakes count extra. Smaller is better.
- **Feature scaling**: rewriting every clue on the same scale, so that a clue measured in small numbers (rain in inches) and one measured in big numbers (temperature in °F) are treated fairly.
- **StandardScaler**: the scikit-learn tool that does the scaling.
- **Ridge**: a straight-line model. Its guess is a starting value plus a **weight** times each clue. A **penalty** keeps the weights from getting too big.
- **Lasso**: like Ridge, but its penalty can push a weight all the way to 0, which drops that clue.
- **Altair**: the Python library that draws our charts. Colab normally has it already.

**Note:** this is a shortened version of the poster analysis (12 clues instead of 20, and straight-line models instead of forests). The "tomorrow = today" row is exactly the same as on the poster. The Ridge row is close to the poster's Ridge row, but not identical, because the poster used more clues.

## Step 1. Get the data

Run the next cell. Click **Choose files** and pick the Excel file. Each row of the file is one county on one day: its AQI and the weather.

In [ ]:
import pandas as pd
import altair as alt
from google.colab import files

uploaded = files.upload()                             # a "Choose files" button appears: pick the Excel file
filename = list(uploaded)[0]                          # the name of the file you uploaded
df = pd.read_excel(filename)                          # read the Excel file into a table called df
df = df.rename(columns={"county Name": "County"})     # a shorter column name
df = df.sort_values(["County", "Date"]).reset_index(drop=True)   # one county after another, oldest day first

print(len(df), "rows |", df.County.nunique(), "counties |", df.Date.min().date(), "to", df.Date.max().date())
df[["Date", "County", "AQI"]].head()                  # the first 5 rows

## Step 2. Make the clues

We want to guess **tomorrow's AQI** in each county. So for every county and day we prepare the **answer** (`tomorrow`) and the **clues** the computer may use:

| Clue | What it is |
|---|---|
| `aqi_today` | today's AQI in the county |
| `aqi_yesterday` | yesterday's AQI |
| `aqi_7_day_average` | the average AQI of the last 7 days |
| `month` | the month number (1 = January ... 12 = December) |
| `county_average` | the county's usual AQI in 2022-2024 |
| 7 weather columns | today's temperature, rain, wind, humidity and sunshine |

The file has other columns too (for example wind direction and the pollutant name). We leave them out to keep things simple. The Data Definition PDF explains every column.

In [ ]:
each_county = df.groupby("County")                    # every calculation below stays inside one county

df["tomorrow"] = each_county.AQI.shift(-1)            # the ANSWER: the AQI on the next row
df["aqi_today"] = df.AQI
df["aqi_yesterday"] = each_county.AQI.shift(1)        # the AQI on the row before
df["aqi_7_day_average"] = each_county.AQI.transform(lambda day: day.rolling(7).mean())   # today and the 6 days before
df["month"] = df.Date.dt.month

# The county's usual AQI. We use 2022-2024 only, so the test days (2025 and later) stay unseen.
usual = df[df.Date < "2025-01-01"].groupby("County").AQI.mean()
df["county_average"] = df.County.map(usual)

In [ ]:
weather = ["Temp Max (°F)", "Temp Mean (°F)", "Precipitation (in)", "Wind Speed Max (mph)",
           "Wind Gust Max (mph)", "Relative Humidity (%)", "Solar Radiation (MJ/m²)"]
clues = ["aqi_today", "aqi_yesterday", "aqi_7_day_average", "month", "county_average"] + weather

# Keep only the days that have every clue, and where the next row really is the next day
next_row_is_tomorrow = (each_county.Date.shift(-1) - df.Date).dt.days == 1
df = df[next_row_is_tomorrow].dropna(subset=clues + ["tomorrow"])
print("Days we can use:", len(df))

## Step 3. Learn from the past, test on the future

We **train** (let the computer learn) on the days before 2025, and we **test** (check its guesses) on 2025 and later. We never mix them, because a real forecast cannot see the future.

In [ ]:
train = df[df.Date < "2025-01-01"]        # 2022-2024: the computer learns from these days
test = df[df.Date >= "2025-01-01"]        # 2025 to May 2026: we check its guesses on these days

X_train, y_train = train[clues], train.tomorrow     # X = the clues, y = the answers
X_test, y_test = test[clues], test.tomorrow
print("Training days:", len(train), "| Test days:", len(test))

## Step 4. Scale the clues (feature scaling)

Our clues come in very different sizes. Rain is usually about 0.1 inch, while temperature is around 60 °F and AQI is around 45.

Ridge and Lasso add a **penalty** that punishes big weights. A clue with small numbers (rain) needs a big weight to matter, so it would be punished more than a clue with big numbers (temperature), only because of its **units**. That is not fair.

**Feature scaling** fixes this. For each clue, **StandardScaler** subtracts the clue's average and divides by its spread (its *standard deviation*). Afterwards every clue has an average of 0 and a spread of 1. A value of +1 means "one step above the usual", and -1 means "one step below the usual", whatever the clue was measured in.

Two rules:
- The scaler learns the averages and spreads from the **training days only**. We then use the same numbers on the test days. Learning them from the test days would be peeking.
- Scaling removes no information. It only changes the units.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=clues)   # learn the averages and spreads, then rescale
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=clues)         # rescale with the SAME numbers: no peeking

sizes = pd.DataFrame({"average before": X_train.mean(), "spread before": X_train.std(),
                      "spread after": X_train_scaled.std()})
print(sizes.round(2))
print("Biggest average after scaling:", round(X_train_scaled.mean().abs().max(), 2))

**What this means**
- Look at the `spread before` column: it is only about 0.24 for rain, but 16 to 18 for temperature and today's AQI. After scaling, every spread is 1.
- After scaling, every clue has an average of 0 (the last line shows the biggest one).

## Step 5. Train Ridge and compare it with the baseline

**Ridge** is a straight-line model. It guesses tomorrow's AQI as a starting value plus a **weight** times each clue. While it learns, it picks the weights that fit the training days well, and its **penalty** (called `alpha`) stops the weights from getting too big.

We compare three ways of guessing:
- **Baseline:** tomorrow = today. No learning at all.
- **Ridge (not scaled):** Ridge on the clues as they were.
- **Ridge (scaled):** Ridge on the scaled clues. This is the one we recommend.

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)                 # alpha is the penalty. 1.0 is the usual starting value.
ridge.fit(X_train_scaled, y_train)       # learn from the SCALED training clues

ridge_no_scaling = Ridge(alpha=1.0)      # the very same model, but fed the clues as they were
ridge_no_scaling.fit(X_train, y_train)
print("Both Ridge models have learned from", len(X_train), "training days.")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

guesses = {"Baseline": X_test.aqi_today,                          # the lazy guess: tomorrow = today
           "Ridge (not scaled)": ridge_no_scaling.predict(X_test),
           "Ridge (scaled)": ridge.predict(X_test_scaled)}

errors = {}
for name, guess in guesses.items():                # grade each guess against the real AQI
    errors[name] = {"MAE": mean_absolute_error(y_test, guess),
                    "RMSE": mean_squared_error(y_test, guess) ** 0.5}

table = pd.DataFrame(errors).T                     # one row per way of guessing
print(table.round(2))
print(f"Ridge (scaled) has an RMSE {1 - table.RMSE['Ridge (scaled)'] / table.RMSE['Baseline']:.0%} lower than the baseline.")

**What this means**
- Both Ridge rows beat the baseline: smaller numbers are better guesses.
- The two Ridge rows are almost the same. On this data, scaling does **not** make Ridge more accurate. (For a plain straight-line model with no penalty, scaling changes nothing at all.)
- So why scale? (1) With a penalty, scaling makes every clue be judged on the same footing. (2) It makes the weights comparable, as Step 6 shows. (3) It is the safe habit: with a bigger penalty, or with Lasso, unscaled clues can give different answers.
- The baseline row is the same as on the poster (MAE 10.27, RMSE 14.57), because it uses the same test days.

## Step 6. Which clues does Ridge rely on?

Because the clues are scaled, the **weights can be compared**. A weight says: when this clue goes up by **one step** (one spread), tomorrow's guess moves by this many AQI points, with the other clues unchanged. A longer bar means more influence.

In [ ]:
weights = pd.Series(ridge.coef_, index=clues)                      # one weight per clue
weights = weights.rename_axis("Clue").reset_index(name="Weight")   # turn it into a table for Altair
weights["Effect"] = (weights.Weight > 0).map({True: "pushes tomorrow up", False: "pushes tomorrow down"})

alt.Chart(weights).mark_bar().encode(
    x=alt.X("Weight:Q", title="Weight (AQI points per 1 step up in the clue)"),
    y=alt.Y("Clue:N", sort="-x", title=None),
    color="Effect:N",
).properties(title="Which clues does Ridge use the most?", width=450, height=300)

**What this means**
- Today's AQI has the longest bar by far. One step in today's AQI is about 18 AQI points, and it moves tomorrow's guess by about 8 points.
- Clues that move together, like Temp Max and Temp Mean, share the credit. So do not read one bar on its own.
- Without scaling, this chart would not be fair: a weight "per inch of rain" and a weight "per °F" are in different units.

## Step 7. Where does it struggle?

The very bad days matter most for health. Let's see how Ridge did on the test days when tomorrow's AQI was above 100.

In [ ]:
guess = pd.Series(ridge.predict(X_test_scaled), index=y_test.index)    # Ridge's guess for every test day

bad_days = y_test > 100                                                # days when tomorrow's AQI was above 100
print("Test days with AQI above 100:", bad_days.sum())
print("Of those, days where Ridge guessed above 100:", (guess[bad_days] > 100).sum())
print("On those days the real AQI averaged", round(y_test[bad_days].mean()), "and Ridge's guess averaged", round(guess[bad_days].mean()))

In [ ]:
# One county, June to August 2025: the real AQI against Ridge's guess
county = "Cuyahoga"                                                # Cleveland. Change this to any county you like.
three_months = test[(test.County == county) & (test.Date >= "2025-06-01") & (test.Date < "2025-09-01")]

compare = pd.DataFrame({"Date": three_months.Date + pd.Timedelta(days=1),   # a guess made today is for tomorrow
                        "Real AQI": three_months.tomorrow,
                        "Ridge's guess": guess[three_months.index]})
compare = compare.melt("Date", var_name="Line", value_name="AQI")      # one column for the numbers, one for which line

alt.Chart(compare).mark_line().encode(
    x=alt.X("Date:T", title=None), y="AQI:Q", color="Line:N",
).properties(title=f"{county} County, June to August 2025", width=650, height=250)

**What this means**
- Ridge follows the normal ups and downs, but it does not see the spikes coming: when the real AQI jumps, its guess stays much lower.
- It also reacts a day late. Ridge leans mostly on today's AQI (Step 6), so the guess line tends to rise just after a peak in the real line, when the real AQI has often already come down.
- The poster's forest models have the same weakness. A model cannot warn about a spike when its clues (today's air and weather) do not contain the warning. Scaling cannot fix that, because it adds no new information.

## Step 8 (extra). Lasso: a model that can drop clues

**Lasso** is Ridge's cousin. Both are straight-line models with a penalty, so both want scaled clues. The difference: Ridge only makes weights smaller, while Lasso can make a weight exactly 0, which means it **drops** that clue. We use a penalty of 0.1 here. A bigger penalty drops more clues.

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)                     # a penalty that can push a weight all the way to 0
lasso.fit(X_train_scaled, y_train)           # scaled clues, so every clue is judged on the same footing

lasso_weights = pd.Series(lasso.coef_, index=clues)
print("Clues Lasso dropped:", list(lasso_weights[lasso_weights == 0].index))
print("Lasso's RMSE:", round(mean_squared_error(y_test, lasso.predict(X_test_scaled)) ** 0.5, 2))

**What this means**
- The clues in the list are the ones Lasso did not need. All the other clues kept a weight.
- Compare Lasso's RMSE with the Ridge (scaled) row in Step 5: they are close, so dropping those clues cost almost nothing.
- In a real project we would choose the penalty with cross-validation on the training days. Here 0.1 is only a simple example.

## Summary in plain English

- **Scaling:** StandardScaler puts every clue on the same scale, using only the training days. On this data it does not change how accurate Ridge is, but it makes the penalty fair and the weights comparable.
- **Guessing tomorrow:** Ridge beats the lazy "same as today" guess. Step 5 shows the numbers.
- **Which clues matter:** today's AQI matters most (Step 6).
- **Limit:** like the poster's forest models, Ridge does not catch the spikes, so it is not a warning system yet (Step 7).
- **Extra:** Lasso drops a few clues and is about as accurate as Ridge (Step 8).
- **Next step:** add next-day weather forecasts and wildfire smoke data to help with the spikes.